In [1]:
import requests
import json
import pandas as pd
import numpy as np
import math
import unpackers

In [2]:
# Configuration
project_name = 'ATAT'
api_url = "https://pandatrack.myjetbrains.com/youtrack/api/"

# load api token
with open('.youtracktoken') as file:
    auth_token = file.readline().strip()

# set request headers
auth_header= {"Authorization": f"Bearer {auth_token}"}

In [3]:

# get project id based on project name
projects_list_fields= {"fields": "id,name"}
projects_list = requests.get(api_url+"admin/projects", params=projects_list_fields, headers=auth_header)
project_id = None
for project in projects_list.json():
    if project_name in project['name']:
        project_id = project['id']
        print(project)
if project_id == None:
    raise ValueError(f"Project name {project_name} does not exist")

{'name': 'ATAT', 'id': '0-9', '$type': 'Project'}


In [4]:
# get list of issues for the project
issues_list_fields={
    "fields": (
        "idReadable,"
        "summary,"
        "description,"
        "created,"
        "updated,"
        "reporter(fullName,email,banned),"
        "updater(fullName,email,banned),"
        "resolved,"
        "numberInProject,"
        "comments("
            "text,"
            "author(email),"
            "created"
            "),"
        "tags("
            "name"
            "),"
        "links("
            "direction,"
            "linkType("
                "directed,"
                "name,"
                "sourceToTarget,"
                "targetToSource"
                "),"
            "issues("
                "id,"
                "idReadable"
                ")"
            "),"
        "customFields("
            "name,"
            "value(name,text,minutes,email,banned)"
            ")"
        ),
        "$skip": 599,
        "$top": 150
    }

# Request Issues from the API
issues_list = requests.get(api_url+"admin/projects/"+project_id+"/issues", params=issues_list_fields, headers=auth_header)
all_issues = issues_list.json()



In [20]:
import csv
all_unpacked_issues = [unpackers.unpack_youtrack_issue(issue) for issue in all_issues]
# header, max_field_counts = unpackers.create_csv_header(all_unpacked_issues)

for i in all_unpacked_issues:
    with open(f'./data/unpacked_issue_{i["Issue Id"]}.json', 'w') as f:
        json.dump(i, f)

test = pd.json_normalize(all_unpacked_issues)
test.to_csv('./data/normalized.csv')
type(test["Sprints"])

def flatten_series_to_columns(value, field_name):
    if isinstance(value, str): 
        return value 
    expanded_dict = {}
    print(type(value))
    for i in range(0, len(value)): 
        expanded_dict[f"{field_name}.{i}"] = value[i]
    print(expanded_dict)
    test1 = pd.DataFrame.from_dict(expanded_dict) 
    return test1

sprintsdf = test["Sprints"].apply(flatten_series_to_columns, args=["Sprints"])
sprintsdf
# PLAN
#1. write a function to use with pd.apply() that will expand series/list values to multiple columns and returning a dataframe
# 2. collect resulting dataframes into a list
# concatenate the list of data frames into a new, fully expanded dataframe.

# for unpacked_issue in all_unpacked_issues:
#     normalized_issue =  pd.json_normalize(unpacked_issue)
#     with open(f'./data/normalized.csv', 'w') as f:
#         json.dump(normalized_issue.to_csv(), f)

#     break

# with open(f'./data/jira_issues_csv.csv', 'w') as csvfile:
#     writer = csv.writer(csvfile, delimiter=",", quoting=csv.QUOTE_ALL)
#     writer.writerow(header)
    
#     for unpacked_issue in all_unpacked_issues:
#         writer.writerow(unpackers.unpacked_issue_to_csv(unpacked_issue, max_field_counts))
#         break

# print("done")

<class 'list'>
{'Sprints.0': 'Backlog', 'Sprints.1': 'ATAT Sprint 28'}


ValueError: If using all scalar values, you must pass an index